# Example: HMM-Powered Backtesting

In this example, we fit a JumpHMM model to synthetic training data, generate regime-switching test paths, and backtest the Session 2 **Cobb-Douglas rebalancing engine** across hundreds of Monte Carlo scenarios. We visualize the distribution of outcomes and identify where the engine succeeds and fails.

> **By the end of this example, you will be able to:**
> * Fit and tune a JumpHMM model from price data using the JumpHMM.jl package
> * Generate hundreds of regime-switching synthetic market paths
> * Backtest the Cobb-Douglas rebalancing engine across all paths and analyze the distribution of outcomes

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

In [ ]:
let
    # Constants -
    global Δt = 1.0 / 252.0;
    global rf = 0.05;
    global N_states = 50;       # number of HMM states
    global ν = 5.0;              # Student-t degrees of freedom
    global B₀ = 10000.0;
    global N_short = 21;
    global N_long = 63;
    global offset = N_short + N_long;
    global n_mc_paths = 100;     # Monte Carlo paths (increase for production)
    global n_trading_days = 336; # offset + 252 trading days

    # Asset universe and SIM parameters (same as Session 2) -
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );
    global start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    println("Setup complete: $(n_mc_paths) MC paths, $(N_states) HMM states, ν = $(ν)")
end

___
## Task 1: Fit and Tune the JumpHMM Model
We generate 5 years of synthetic "training" prices (using GBM as a stand-in for historical data), then fit a JumpHMM model and tune its jump parameters to match the training data's kurtosis and autocorrelation structure.

> **What should you see?** The fitted model will have a transition matrix that shows regime persistence (strong diagonal), emission distributions that vary across states (low-volatility calm states vs. high-volatility crisis states), and tuned jump parameters (ε, λ) that produce realistic volatility clustering.

In [ ]:
let
    # Generate 5 years of training data -
    training_prices = generate_training_prices(S₀=100.0, μ=0.08, σ=0.18, T=1260, seed=42);

    # Fit JumpHMM -
    println("Fitting JumpHMM model (N = $(N_states) states)...")
    global hmm_model = hmm_fit(JumpHiddenMarkovModel, training_prices; 
        rf=rf, N=N_states, nu=ν, dt=Δt);
    
    # Tune jump parameters -
    println("Tuning jump parameters (ε, λ)...")
    global hmm_model = hmm_tune(hmm_model, training_prices;
        epsilon_range=range(1e-4, 2.5e-2, length=10),
        lambda_range=range(10.0, 100.0, length=8),
        n_paths=50);
    
    # Report -
    println("\nModel fitted and tuned:")
    println("  Jump probability (ε): $(round(hmm_model.jump.epsilon, digits=5))")
    println("  Jump duration mean (λ): $(round(hmm_model.jump.lambda, digits=1))")
    println("  Number of states: $(N_states)")
    println("  Training data: $(length(training_prices)) days (5 years)")

    # Save model -
    save(joinpath(_PATH_TO_DATA, "hmm-model.jld2"), Dict("model" => hmm_model));
end

**Visualize:** A few sample HMM-generated price paths vs. the GBM training data.

> **What should you see?** The HMM paths should look qualitatively different from GBM — they'll exhibit periods of low volatility punctuated by sudden, persistent drops or rallies (the jump mechanism). Some paths will show severe drawdowns that GBM would never produce.

In [ ]:
let
    # Generate a few sample paths for visualization -
    n_sample = 5;
    P₀ = 100.0;

    p = plot(size=(800, 450), title="Sample HMM-Generated Market Paths",
        xlabel="Trading Day", ylabel="Price (\$)", legend=:topright, alpha=0.8)

    for i in 1:n_sample
        result = hmm_simulate(hmm_model, n_trading_days; n_paths=1);
        G = result.paths[1].observations;
        prices = JumpHMM.prices_from_growth_rates(G, P₀; rf=rf, dt=Δt);
        plot!(p, 1:length(prices), prices, label=(i == 1 ? "HMM paths" : ""), 
            linewidth=1.2, color=:steelblue, alpha=0.5)
    end

    # Add a GBM reference path -
    gbm_prices = generate_training_prices(S₀=P₀, μ=0.08, σ=0.18, T=n_trading_days, seed=99);
    plot!(p, 1:length(gbm_prices), gbm_prices, label="GBM reference", 
        linewidth=2, color=:coral, linestyle=:dash)
    
    vline!(p, [offset], label="Warmup end", linestyle=:dot, color=:black, alpha=0.5)
    p
end

___
## Task 2: Generate the Backtest Scenario
We use the fitted HMM to generate a full backtest scenario — $(n\_mc\_paths)$ market paths, each with per-ticker prices generated via SIM. This creates the Monte Carlo test bed for strategy evaluation.

> **What should you see?** A fan of price paths showing the range of possible outcomes. The paths should exhibit diverse behaviors — some bullish, some crashing, some sideways — reflecting the full regime-switching dynamics of the HMM.

In [ ]:
let
    println("Generating backtest scenario: $(n_mc_paths) paths × $(n_trading_days) days...")

    global scenario = generate_hmm_scenario(hmm_model, tickers, sim_params;
        n_paths=n_mc_paths, n_steps=n_trading_days, P₀_market=100.0,
        start_prices=start_prices, rf=rf, Δt=Δt, label="HMM Regime-Switching");

    println("Scenario generated: $(scenario.n_paths) paths × $(scenario.n_steps) days × $(length(tickers)) tickers")

    # Save for Example 2 -
    save(joinpath(_PATH_TO_DATA, "backtest-scenario.jld2"), Dict("scenario" => scenario));
end

___
## Task 3: Backtest the Cobb-Douglas Engine Across All Paths
We run the Session 2 Cobb-Douglas rebalancing engine (with moderate trigger rules: DD ≤ 15%, τ ≤ 50%) and the buy-and-hold benchmark across all Monte Carlo paths. We collect the distribution of final wealth, max drawdown, and Sharpe ratio.

> **What should you see?** Histograms showing the spread of outcomes. The engine should have a tighter distribution (less variance) than buy-and-hold due to its trigger rules, but some paths will still show losses — the question is how often and how bad.

In [ ]:
let
    # Backtest the Cobb-Douglas engine -
    println("Backtesting Cobb-Douglas Engine across $(n_mc_paths) paths...")
    rules_params = (max_drawdown = 0.15, max_turnover = 0.50);
    global engine_bt = backtest_engine(scenario, tickers, sim_params, rules_params;
        B₀=B₀, offset=offset);

    # Backtest buy-and-hold -
    println("Backtesting Buy-and-Hold across $(n_mc_paths) paths...")
    global bh_bt = backtest_buyhold(scenario, tickers; B₀=B₀, offset=offset);

    # Summary statistics -
    println("\n" * "═"^55)
    println("  BACKTEST RESULTS: $(n_mc_paths) HMM paths")
    println("═"^55)

    summary = DataFrame(
        "Metric" => ["Median Final Wealth", "Median Max Drawdown (%)", 
            "Median Sharpe", "Failure Rate (% below \$$(Int(B₀)))"],
        "Cobb-Douglas Engine" => [
            round(median(engine_bt.final_wealth), digits=0),
            round(median(engine_bt.max_drawdowns) * 100, digits=1),
            round(median(engine_bt.sharpe_ratios), digits=3),
            round(sum(engine_bt.final_wealth .< B₀) / n_mc_paths * 100, digits=1)
        ],
        "Buy-and-Hold" => [
            round(median(bh_bt.final_wealth), digits=0),
            round(median(bh_bt.max_drawdowns) * 100, digits=1),
            round(median(bh_bt.sharpe_ratios), digits=3),
            round(sum(bh_bt.final_wealth .< B₀) / n_mc_paths * 100, digits=1)
        ]
    );
    pretty_table(summary, tf=tf_markdown)

    # Save for Example 2 -
    save(joinpath(_PATH_TO_DATA, "backtest-results.jld2"),
        Dict("engine" => engine_bt, "buyhold" => bh_bt));
end

**Visualize:** Distribution of final wealth and max drawdown for both strategies.

> **What should you see?** Two overlapping histograms. The engine's distribution should be shifted right (higher wealth) and/or have a thinner left tail (fewer catastrophic losses) compared to buy-and-hold. The drawdown distribution should show the engine's trigger rules capping extreme losses.

In [ ]:
let
    p1 = histogram(engine_bt.final_wealth, label="Cobb-Douglas Engine", alpha=0.6, color=:steelblue, bins=20)
    histogram!(p1, bh_bt.final_wealth, label="Buy-and-Hold", alpha=0.4, color=:coral, bins=20)
    vline!(p1, [B₀], label="Starting Capital", linestyle=:dash, color=:black, linewidth=2)
    xlabel!(p1, "Final Wealth (\$)")
    ylabel!(p1, "Count")
    title!(p1, "Distribution of Final Wealth")

    p2 = histogram(engine_bt.max_drawdowns .* 100, label="Cobb-Douglas Engine", alpha=0.6, color=:steelblue, bins=20)
    histogram!(p2, bh_bt.max_drawdowns .* 100, label="Buy-and-Hold", alpha=0.4, color=:coral, bins=20)
    xlabel!(p2, "Max Drawdown (%)")
    ylabel!(p2, "Count")
    title!(p2, "Distribution of Max Drawdown")

    plot(p1, p2, layout=(1, 2), size=(900, 400), legend=:topright)
end

___
## Summary

> __Key Takeaways:__
>
> * **JumpHMM fitting is a two-step process**: First fit the model to training data, then tune jump parameters ($\epsilon$, $\lambda$) to match empirical kurtosis and autocorrelation — the resulting model generates paths with realistic regime transitions
> * **Distributional backtesting exposes tail risk**: Running the Cobb-Douglas engine across hundreds of HMM paths reveals the spread of terminal wealth, drawdown, and Sharpe outcomes — including the worst-case scenarios that matter most for risk management
> * **The engine handles regime shifts but has limits**: Performance degrades during sustained crisis regimes, motivating the bandit challenger we introduce in the next example

In the next example, we introduce the **bandit challenger** and produce the formal validation report.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.